### Collect/prepare data

In [ ]:
import os
os.makedirs('./data', exist_ok=True)

In [ ]:
!wget -O ./data/MatPES-PBE-2025.1.json.gz https://s3.us-east-1.amazonaws.com/materialsproject-contribs/MatPES_2025_1/MatPES-PBE-2025.1.json.gz
!wget -O ./data/MatPES-PBE-split.json.gz https://s3.us-east-1.amazonaws.com/materialsproject-contribs/MatPES_2025_1/MatPES-PBE-split.json.gz

In [ ]:
!wget -O ./data/MatPES-PBE-atoms.json.gz https://s3.us-east-1.amazonaws.com/materialsproject-contribs/MatPES_2025_1/MatPES-PBE-atoms.json.gz

In [ ]:
from tqdm import tqdm
from monty.serialization import loadfn

pbe = loadfn("./data/MatPES-PBE-2025.1.json.gz")
splits = loadfn("./data/MatPES-PBE-split.json.gz")

train_set = []
valid_set = []
test_set = []

for i, d in tqdm(enumerate(pbe)):
    atoms = AseAtomsAdaptor.get_atoms(d['structure'])
    atoms.info['total_energy_matpes_pbe'] = d['energy']
    atoms.info['energy_per_atom_matpes_pbe'] = d['energy']/d['nsites']
    atoms.info['forces_matpes_pbe'] = d['forces']
    atoms.info['stress_matpes_pbe'] = d['stress']
    if i in splits["train"]:
        train_set.append(atoms)
    elif i in splits["valid"]:
        valid_set.append(atoms)
    else:
        test_set.append(atoms)

print(f"{len(train_set)}-{len(valid_set)}-{len(test_set)}")
# Output is 391240-21735-21737

In [ ]:
write('./data/MatPES-PBE-2025.1.train.xyz', train_set)
write('./data/MatPES-PBE-2025.1.valid.xyz', valid_set)
write('./data/MatPES-PBE-2025.1.test.xyz', test_set)

### Train model

In [ ]:
from ase.data import atomic_numbers
from monty.serialization import loadfn

atoms_ref_data = loadfn("./data/MatPES-PBE-atoms.json.gz")

E0s = {}
for data in atoms_ref_data:
    try:
        idx = atomic_numbers[data['elements'][0]]
        E0s[idx] = data['energy']
    except:
        continue
for i in range(128):
    if i not in E0s.keys():
        E0s[i] = 0.0
        
E0s_sorted = dict(sorted(E0s.items()))

In [ ]:
from gnn_lib.models import CrystalGraphConvNet
from gnn_lib.data_utils import build_dataloader
from gnn_lib.config import Config
from gnn_lib.training import Trainer

config = Config.from_file('../configs/cgcnn_matpes_pbe_small.yaml')
config.model.E0s = E0s_sorted

In [ ]:
model = CrystalGraphConvNet(**config.model)
model.size()

In [ ]:
train_loader = build_dataloader(config, 'train')
val_loader = build_dataloader(config, 'val')

In [ ]:
trainer = Trainer(model, config, verbose=True)
trainer.train(train_loader, val_loader)

### Evaluate model

In [ ]:
import torch
from tqdm import tqdm
from gnn_lib.models import CrystalGraphConvNet
from gnn_lib.data_utils import build_dataloader
from gnn_lib.config import Config


device = 'cuda'
config = Config.from_file('processed_data_matpes_pbe_small/config.yaml')
model = CrystalGraphConvNet(**config.model)
model.from_checkpoint('checkpoints_matpes_pbe_small/best_checkpoint.pt')

config.model.E0s = None # 
test_loader = build_dataloader(config, 'test')


e_preds, e_labels = [], []
f_preds, f_labels = [], []
s_preds, s_labels = [], []

model.to(device)
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        
        out = model(batch.to(device))

        e = out['energy']
        e_raw = model._energy_from_e0s_and_residuals(e, batch)

        e_preds.extend(e_raw.detach().cpu().numpy().flatten())        
        e_labels.extend(batch.energy.detach().cpu().numpy().flatten())
        
        
        f_preds.extend(out['forces'].detach().cpu().numpy().flatten())
        f_labels.extend(batch.forces.detach().cpu().numpy().flatten())
        
        s_preds.extend(1602.1766 * out['stress'].detach().cpu().numpy().flatten())
        s_labels.extend(batch.stress.detach().cpu().numpy().flatten())

In [ ]:
import numpy as np

e_preds = np.stack(e_preds)
e_labels = np.stack(e_labels)

f_preds = np.stack(f_preds)
f_labels = np.stack(f_labels)

s_preds = np.stack(s_preds)
s_labels = np.stack(s_labels)

In [ ]:
from gnn_lib.metrics import get_metrics

energy_metrics = get_metrics(e_labels, e_preds)

In [ ]:
forces_metrics = get_metrics(f_labels, f_preds)

In [ ]:
stress_metrics = get_metrics(s_labels, s_preds)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(dpi = 150, figsize=(8, 4), ncols=2)

ax1.scatter(e_labels, e_preds, alpha = 1.0, s = 1, label='Data')
ax1.plot(e_labels, e_labels, color ='k', label = 'Perfect prediction')
ax1.set_xlabel('Energy (DFT), eV/atom')
ax1.set_ylabel('Energy (model), eV/atom')
ax1.set_xlim(min(e_labels.min(), e_preds.min()), max(e_labels.max(), e_preds.max()))
ax1.set_ylim(min(e_labels.min(), e_preds.min()), max(e_labels.max(), e_preds.max()))
ax1.legend()

_ = ax2.hist(e_labels - e_preds, bins=500)
ax2.set_xlim(-0.5, 0.5)
ax2.set_xlabel('Model error, eV/atom')
ax2.set_ylabel('Count')

plt.tight_layout()

In [ ]:
#torch.save(model, 'cgcnn_matpes_pbe_small_model.pth')
#model = torch.load('cgcnn_matpes_pbe_small_model.pth', weights_only=False)